## Expression rules

In [1]:
import Pkg; Pkg.add("ExprRules")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [2]:
import Pkg; Pkg.add("TreeView")


   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [3]:
import Pkg; Pkg.add("Distributions")


   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [4]:
using ExprRules
using TreeView
using Distributions
using Random


In [5]:
grammar

UndefVarError: UndefVarError: `grammar` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [37]:
# Example 20.3. Example of defining a grammar using the ExprRules.jl package.
grammar = @grammar begin

    R = R * A # multiple children
    R = f(R) # call a function
    R =_(randn()) # random variable generated on node creation
    # R = 1 | 2 | 3 # equivalent to R = 1, R = 2, and R = 3
    # R = |(4:6) # equivalent to R = 4, R = 5, and R = 6
    A = 7 # rules for different return types
end;

f(x) = 2x

f (generic function with 1 method)

In [38]:
grammar

1: R = R * A
2: R = f(R)
3: R = _(randn())
4: A = 7


In [39]:
rand(RuleNode, grammar, :R,6)

3,

In [40]:
[get_executable(rand(RuleNode, grammar, :R,6),  grammar)  for i in 1:10]

10-element Vector{Any}:
   :(f(2.1962896199539745))
   :(f(f(-0.41013937074120443) * 7))
   :(1.0840946687653166 * 7)
   :(f(0.3769767230737012))
 -0.43868474739326524
   :(f(1.08223640224734))
   :(f(f((f(0.02158565441525684) * 7) * 7)))
   :(f(f(-0.4559772290663146 * 7)))
  0.6276978446151886
  0.5779625404373697

In [41]:
expr = rand(RuleNode, grammar, :R,40)

3,

In [42]:
get_executable(expr,  grammar)

0.72562918963945

In [43]:

Symbols = SymbolTable(grammar)

Dict{Symbol, Any} with 2 entries:
  :f => f
  :* => *

In [44]:
Core.eval(Symbols, get_executable(expr,  grammar) )

0.72562918963945

## Genetic programming

In [46]:
abstract type SelectionMethod end
struct TruncationSelection <: SelectionMethod
    k # top k to keep
end
function select(t::TruncationSelection, y)
    p = sortperm(y)
    return [p[rand(1:t.k, 2)] for i in y]

end

select (generic function with 1 method)

In [47]:
abstract type CrossoverMethod end
struct TreeCrossover <: CrossoverMethod
    grammar
    max_depth
end
function crossover(C::TreeCrossover, a, b)
    child = deepcopy(a)
    crosspoint = sample(b)
    typ = return_type(C.grammar, crosspoint.ind)
    d_subtree = depth(crosspoint)
    d_max = C.max_depth + 1 - d_subtree
    if d_max > 0 && contains_returntype(child, C.grammar, typ, d_max)
        loc = sample(NodeLoc, child, typ, C.grammar, d_max)
        insert!(child, loc, deepcopy(crosspoint))
    end
    child
end

crossover (generic function with 1 method)

In [48]:
expr=rand(RuleNode, grammar, :R,4)
cr= sample(expr)
expr,cr, cr.ind, get_executable(expr, grammar),get_executable(cr, grammar), return_type(grammar, cr.ind)

(3,, 3,, 3, -1.1723317923460677, -1.1723317923460677, :R)

In [49]:
abstract type MutationMethod end
struct TreeMutation <: MutationMethod
    grammar
    p
end
function mutate(M::TreeMutation, a)
    child = deepcopy(a)
    if rand() < M.p
        loc = sample(NodeLoc, child)
        typ = return_type(M.grammar, get(child, loc).ind)
        subtree = rand(RuleNode, M.grammar, typ)
        insert!(child, loc, subtree)
    end
    return child

end

mutate (generic function with 2 methods)

In [50]:
struct TreePermutation <: MutationMethod
    grammar
    p
end
function mutate(M::TreePermutation, a)
    child = deepcopy(a)
    if rand() < M.p
        node = sample(child)
        n = length(node.children)
        types = child_types(M.grammar, node)
        for i in 1:n-1
            c = 1
            for k in i+1:n
                if types[k] == types[i] &&
                   rand() < 1 / (c += 1)
                    node.children[i], node.children[k] =
                        node.children[k], node.children[i]

                end
            end
            
        end
    end
    return child
end

mutate (generic function with 2 methods)

In [51]:
function genetic_algorithm(f, population, max_iter, selection, crossover_, mutation)
    m = length(population)
    n = length(population[1])
    y = [f(population[i]) for i in 1:m]
    for i in 1:max_iter
        parents = select(selection, y)
        
        children = [crossover(crossover_, population[p[1]], population[p[2]]) for p in parents]
        children = [mutate(mutation, c) for c in children]
        children_y = [f(c) for c in children]
        for j in 1:m
            if children_y[j] < y[j]
                y[j] = children_y[j]
                population[j] = children[j]
            end
        end
        # population = children
        # y = children_y
        @show minimum(y)
    end
    return population[argmin(y)]
    
end

genetic_algorithm (generic function with 1 method)

In [52]:
grammar = @grammar begin
    R = |(1:9)
    R = R + R
    R = R - R
    R = R / R
    R = R * R
end

function f(node)
    value = Core.eval(node, grammar)
    if isinf(value) || isnan(value)
        return Inf
    end
    Δ = abs(value - π)
    return log(Δ) + length(node) / 1e3
end


population = [rand(RuleNode, grammar, :R) for i in 1:1000]
best_tree = genetic_algorithm(f, population, 30,
    TruncationSelection(50),
    TreeCrossover(grammar, 10),
    TreeMutation(grammar, 0.25))
    # TreePermutation(grammar, 0.25))
get_executable(best_tree, grammar)

minimum(y) = -4.7814129882422876
minimum(y) = -4.7814129882422876
minimum(y) = -4.7814129882422876
minimum(y) = -5.892110143483638
minimum(y) = -5.892110143483638
minimum(y) = -5.892110143483638
minimum(y) = -5.892110143483638
minimum(y) = -5.892110143483638
minimum(y) = -5.892110143483638
minimum(y) = -5.892110143483638
minimum(y) = -5.892110143483638
minimum(y) = -5.892110143483638
minimum(y) = -5.894110143483638
minimum(y) = -5.894110143483638
minimum(y) = -5.894110143483638
minimum(y) = -5.894110143483638
minimum(y) = -5.894110143483638
minimum(y) = -5.894110143483638
minimum(y) = -5.894110143483638
minimum(y) = -5.894110143483638
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295
minimum(y) = -5.894110143484295


:(3 - ((1 / (8 * 3) + 9) * 2 - 2 * (1 / 9 + 9)))

In [53]:
Core.eval(best_tree, grammar)

3.1388888888888893

In [54]:
π

π = 3.1415926535897...

## TO DO



- Analize the code below;
- Modify the code to learn the function $x^3+\sin(x)-3x+7$;


In [55]:
grammar = @grammar begin
    R = x
    R = R * R
    R = R + R
    R = R - R
    R = |(1:5)
    R = sin(x)
end

1: R = x
2: R = R * R
3: R = R + R
4: R = R - R
5: R = 1
6: R = 2
7: R = 3
8: R = 4
9: R = 5
10: R = sin(x)


In [58]:
const S = SymbolTable(grammar)
ground_truth(x) = x*x + 2x + 1

function loss(node)
    ex = get_executable(node, grammar)
    los = 0.0
    for x = -5.:1.:5.
        S[:x] = x
        los += abs2(Core.eval(S,ex) - ground_truth(x))
    end
    los + length(node)/1000
end

loss (generic function with 1 method)

In [59]:
population = [rand(RuleNode, grammar, :R) for i in 1:1000]
best_tree = genetic_algorithm(loss, population, 100,
    TruncationSelection(50),
    TreeCrossover(grammar, 10),
    TreeMutation(grammar, 0.25))
    # TreePermutation(grammar, 0.25))
get_executable(best_tree, grammar)

minimum(y) = 44.009
minimum(y) = 11.007
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y) = 0.011
minimum(y)

:(3 + ((x * x + x) - (2 - x)))